# DAPT-BERT Experiment Runner

This notebook runs the two domain-adaptive pretraining (DAPT) experiments for `bert-base-uncased`:

| Experiment | MLM corpus | Fine-tuning |
|---|---|---|
| BERT DAPT | FPB + TFNS train splits | FPB, TFNS |
| BERT DAPT (augmented) | `financial_mlm_corpus_20000.csv` | FPB, TFNS |

Pipeline:
1. Clone repo and set up environment
2. Mount Google Drive
3. Install dependencies
4. Check required files
5. BERT DAPT: MLM pretraining on FPB + TFNS
6. BERT DAPT: Fine-tuning on FPB
7. BERT DAPT: Fine-tuning on TFNS
8. BERT DAPT (augmented): MLM pretraining on financial corpus
9. BERT DAPT (augmented): Fine-tuning on FPB
10. BERT DAPT (augmented): Fine-tuning on TFNS
11. Collect and summarise all results
12. Filter to test-set results
13. Compute generalisation drop

## Step 1: Clone the Repository

In [2]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/huaijin090906-collab/financial-sentiment-project.git
    %cd financial-sentiment-project
    !pwd
    !ls
else:
    print("Running locally — skipping clone. Make sure you are running this notebook from inside the repo directory.")

Cloning into 'financial-sentiment-project'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 269 (delta 149), reused 164 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (269/269), 3.80 MiB | 14.92 MiB/s, done.
Resolving deltas: 100% (149/149), done./149)
/content/financial-sentiment-project/financial-sentiment-project
/content/financial-sentiment-project/financial-sentiment-project
configs  data  notebooks  README.md  requirements.txt  scripts	src


## Step 2: Setup Environment and Project Root

In [3]:
from pathlib import Path
import os
import sys
import json
import subprocess
import platform
import shlex

import pandas as pd
import yaml

def find_project_root(start=None):
    sentinels = ("README.md", "requirements.txt")
    current = (start or Path.cwd()).resolve()
    candidates = [current] + list(current.parents)
    for candidate in candidates:
        if all((candidate / s).exists() for s in sentinels):
            return candidate
    raise FileNotFoundError("Could not locate project root from current notebook location.")

ROOT = find_project_root()
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

IN_COLAB = "google.colab" in sys.modules

print(f"Project root: {ROOT}")
print(f"Python: {sys.executable}")
print(f"Platform: {platform.platform()}")
print(f"In Colab: {IN_COLAB}")
print(f"Current working directory: {Path.cwd()}")

Project root: /content/financial-sentiment-project/financial-sentiment-project
Python: /usr/bin/python3
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
In Colab: True
Current working directory: /content/financial-sentiment-project/financial-sentiment-project


## Step 3: Mount Google Drive

In [4]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
else:
    print("Not running in Colab. Skipping Drive mount.")

Mounted at /content/drive
Google Drive mounted.


## Step 4: Install Dependencies

In [5]:
if IN_COLAB:
    cmd = [sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(ROOT), text=True)
    if result.returncode != 0:
        raise RuntimeError("Dependency installation failed.")
    print("Dependencies are ready.")
else:
    print("Running locally — skipping pip install. Assuming requirements already installed.")

$ /usr/bin/python3 -m pip install -r /content/financial-sentiment-project/financial-sentiment-project/requirements.txt
Dependencies are ready.


## Step 5: Helper Functions

In [6]:
def snapshot_paths(pattern: str) -> set[str]:
    return {str(p.resolve()) for p in ROOT.glob(pattern)}


def newly_created_paths(before: set[str], pattern: str) -> list[Path]:
    after = snapshot_paths(pattern)
    new_items = [Path(p) for p in (after - before)]
    return sorted(new_items, key=lambda p: p.stat().st_mtime)


def newest_path(pattern: str):
    matches = sorted(ROOT.glob(pattern), key=lambda p: p.stat().st_mtime)
    return matches[-1] if matches else None


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def flatten_metrics_json(metrics_path: Path) -> pd.DataFrame:
    payload = load_json(metrics_path)

    model_name = payload.get("model_name") or payload.get("model_type") or "unknown"
    run_name = payload.get("run_name", "")
    train_path = payload.get("train_path")
    train_domain = Path(train_path).parent.name if train_path else "none"

    rows = []
    for eval_name, eval_payload in payload.get("eval_sets", {}).items():
        eval_path = eval_payload.get("path", "")
        eval_domain = Path(eval_path).parent.name if eval_path else "unknown"
        eval_split = Path(eval_path).stem if eval_path else eval_name

        rows.append(
            {
                "run_name": run_name,
                "model": model_name,
                "train_domain": train_domain,
                "eval_name": eval_name,
                "eval_domain": eval_domain,
                "eval_split": eval_split,
                "setting": (
                    "lexicon_eval"
                    if train_domain == "none"
                    else "in_domain" if train_domain == eval_domain else "cross_domain"
                ),
                "accuracy": eval_payload.get("accuracy"),
                "macro_f1": eval_payload.get("macro_f1"),
                "weighted_f1": eval_payload.get("weighted_f1"),
                "metrics_path": str(metrics_path),
                "prediction_path": eval_payload.get("prediction_path"),
                "confusion_matrix_path": eval_payload.get("confusion_matrix_path"),
                "confusion_figure_path": eval_payload.get("confusion_figure_path"),
            }
        )

    return pd.DataFrame(rows)


def collect_dapt_metrics() -> pd.DataFrame:
    """Collect metrics from the two DAPT-BERT output directories."""
    patterns = [
        "outputs/metrics/mlm_finetune_bert/*.json",
        "outputs/metrics/mlm_finetune_bert_augmentation/*.json",
    ]
    metric_files = []
    for pattern in patterns:
        for path in ROOT.glob(pattern):
            try:
                payload = load_json(path)
                if "eval_sets" in payload:
                    metric_files.append(path)
            except Exception:
                continue

    frames = [flatten_metrics_json(path) for path in metric_files]
    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True)
    combined = combined.sort_values(["model", "run_name", "eval_name"]).reset_index(drop=True)
    return combined


def make_temp_mlm_ft_config(src_config_path: Path, checkpoint_dir: Path, suffix: str) -> Path:
    with src_config_path.open("r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    config["model"]["name"] = str(checkpoint_dir)

    out_dir = ROOT / "outputs/tmp_configs"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"{src_config_path.stem}_{suffix}.yaml"
    with out_path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)

    print(f"Temporary config written to: {out_path}")
    return out_path


def run_live_command(cmd: list, cwd=None) -> None:
    cwd = cwd or ROOT
    shell_cmd = " ".join(shlex.quote(str(x)) for x in cmd)
    full_cmd = (
        f'cd {shlex.quote(str(cwd))} && '
        f'PROJECT_ROOT={shlex.quote(str(ROOT))} '
        f'PYTHONUNBUFFERED=1 '
        f'{shell_cmd}'
    )
    print("$", full_cmd)
    exit_code = get_ipython().system(full_cmd)
    if exit_code not in (0, None):
        raise RuntimeError(f"Command failed with exit code {exit_code}")


print("Helper functions ready.")

Helper functions ready.


## Step 6: Check Required Files

In [7]:
required_files = [
    ROOT / "configs/mlm_pretrain_bert.yaml",
    ROOT / "configs/transformer_bert_mlm_fpb.yaml",
    ROOT / "configs/transformer_bert_mlm_tfns.yaml",
    ROOT / "configs/mlm_pretrain_bert_augmentation.yaml",
    ROOT / "configs/transformer_bert_mlm_fpb_augmentation.yaml",
    ROOT / "configs/transformer_bert_mlm_tfns_augmentation.yaml",
    ROOT / "data/processed/fpb/train.csv",
    ROOT / "data/processed/fpb/val.csv",
    ROOT / "data/processed/fpb/test.csv",
    ROOT / "data/processed/tfns/train.csv",
    ROOT / "data/processed/tfns/val.csv",
    ROOT / "data/processed/tfns/test.csv",
    ROOT / "data/financial_mlm_corpus_20000.csv",
]

status_rows = []
for path in required_files:
    status_rows.append({"path": str(path.relative_to(ROOT)), "exists": path.exists()})

status_df = pd.DataFrame(status_rows)
display(status_df)

missing = status_df.loc[~status_df["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")
else:
    print("All required files are present.")

,path,exists
0,configs/mlm_pretrain_bert.yaml,True
1,configs/transformer_bert_mlm_fpb.yaml,True
2,configs/transformer_bert_mlm_tfns.yaml,True
3,configs/mlm_pretrain_bert_augmentation.yaml,True
4,configs/transformer_bert_mlm_fpb_augmentation....,True
5,configs/transformer_bert_mlm_tfns_augmentation...,True
6,data/processed/fpb/train.csv,True
7,data/processed/fpb/val.csv,True
8,data/processed/fpb/test.csv,True
9,data/processed/tfns/train.csv,True


All required files are present.


## BERT DAPT: Continued MLM Pretraining + Fine-Tuning

Uses `bert-base-uncased` as the base model. MLM continued pretraining is performed
on the FPB + TFNS training splits (same in-domain corpus as the FinBERT DAPT baseline).
Expected ~20-30 min on a T4 GPU.

In [8]:
before_ckpts = snapshot_paths("outputs/checkpoints/mlm_pretrain_bert/*_best")

run_live_command([
    sys.executable,
    "-m",
    "src.cli.run_mlm_pretrain",
    "--config",
    str(ROOT / "configs/mlm_pretrain_bert.yaml"),
])

new_ckpts = newly_created_paths(before_ckpts, "outputs/checkpoints/mlm_pretrain_bert/*_best")
mlm_bert_checkpoint_dir = new_ckpts[-1] if new_ckpts else newest_path("outputs/checkpoints/mlm_pretrain_bert/*_best")

print(f"BERT MLM checkpoint directory: {mlm_bert_checkpoint_dir}")
if mlm_bert_checkpoint_dir is None:
    raise FileNotFoundError("Could not find BERT MLM checkpoint directory after pretraining.")

$ cd /content/financial-sentiment-project/financial-sentiment-project && PROJECT_ROOT=/content/financial-sentiment-project/financial-sentiment-project PYTHONUNBUFFERED=1 /usr/bin/python3 -m src.cli.run_mlm_pretrain --config /content/financial-sentiment-project/financial-sentiment-project/configs/mlm_pretrain_bert.yaml
BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Starting: MLM Pretraining
Training config: epochs=5, batch_size=16, lr=5.00e-05
MLM Pretraining | epoch 1/5 | step 571/571 | total 571/2855 |

In [9]:
if "mlm_bert_checkpoint_dir" not in globals() or mlm_bert_checkpoint_dir is None:
    mlm_bert_checkpoint_dir = newest_path("outputs/checkpoints/mlm_pretrain_bert/*_best")

if mlm_bert_checkpoint_dir is None:
    raise FileNotFoundError("No BERT MLM checkpoint found. Run the pretraining cell first.")

temp_bert_mlm_fpb_config = make_temp_mlm_ft_config(
    ROOT / "configs/transformer_bert_mlm_fpb.yaml",
    mlm_bert_checkpoint_dir,
    "auto",
)

temp_bert_mlm_tfns_config = make_temp_mlm_ft_config(
    ROOT / "configs/transformer_bert_mlm_tfns.yaml",
    mlm_bert_checkpoint_dir,
    "auto",
)

print("BERT MLM fine-tuning configs ready.")
print(temp_bert_mlm_fpb_config)
print(temp_bert_mlm_tfns_config)

Temporary config written to: /content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_fpb_auto.yaml
Temporary config written to: /content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_tfns_auto.yaml
BERT MLM fine-tuning configs ready.
/content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_fpb_auto.yaml
/content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_tfns_auto.yaml


In [10]:
before = snapshot_paths("outputs/metrics/mlm_finetune_bert/*.json")

run_live_command([
    sys.executable,
    "-m",
    "src.cli.train_transformer",
    "--config",
    str(temp_bert_mlm_fpb_config),
])

new_metric_files = newly_created_paths(before, "outputs/metrics/mlm_finetune_bert/*.json")
bert_mlm_fpb_metrics_path = new_metric_files[-1] if new_metric_files else newest_path("outputs/metrics/mlm_finetune_bert/*.json")

print(f"BERT MLM+FT (FPB) metrics file: {bert_mlm_fpb_metrics_path}")
display(flatten_metrics_json(bert_mlm_fpb_metrics_path))

$ cd /content/financial-sentiment-project/financial-sentiment-project && PROJECT_ROOT=/content/financial-sentiment-project/financial-sentiment-project PYTHONUNBUFFERED=1 /usr/bin/python3 -m src.cli.train_transformer --config /content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_fpb_auto.yaml
BertForSequenceClassification LOAD REPORT from: /content/financial-sentiment-project/financial-sentiment-project/outputs/checkpoints/mlm_pretrain_bert/mlm_pretrain_bert_20260416_104251_best
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING

,run_name,model,train_domain,eval_name,eval_domain,eval_split,setting,accuracy,macro_f1,weighted_f1,metrics_path,prediction_path,confusion_matrix_path,confusion_figure_path
0,bert_mlm_fpb_20260416_105113,/content/financial-sentiment-project/financial...,fpb,fpb_val,fpb,val,in_domain,0.976401,0.966415,0.976586,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
1,bert_mlm_fpb_20260416_105113,/content/financial-sentiment-project/financial...,fpb,fpb_test,fpb,test,in_domain,0.979351,0.975231,0.979477,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
2,bert_mlm_fpb_20260416_105113,/content/financial-sentiment-project/financial...,fpb,tfns_test,tfns,test,cross_domain,0.720339,0.672261,0.728330,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...


In [11]:
before = snapshot_paths("outputs/metrics/mlm_finetune_bert/*.json")

run_live_command([
    sys.executable,
    "-m",
    "src.cli.train_transformer",
    "--config",
    str(temp_bert_mlm_tfns_config),
])

new_metric_files = newly_created_paths(before, "outputs/metrics/mlm_finetune_bert/*.json")
bert_mlm_tfns_metrics_path = new_metric_files[-1] if new_metric_files else newest_path("outputs/metrics/mlm_finetune_bert/*.json")

print(f"BERT MLM+FT (TFNS) metrics file: {bert_mlm_tfns_metrics_path}")
display(flatten_metrics_json(bert_mlm_tfns_metrics_path))

$ cd /content/financial-sentiment-project/financial-sentiment-project && PROJECT_ROOT=/content/financial-sentiment-project/financial-sentiment-project PYTHONUNBUFFERED=1 /usr/bin/python3 -m src.cli.train_transformer --config /content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_tfns_auto.yaml
BertForSequenceClassification LOAD REPORT from: /content/financial-sentiment-project/financial-sentiment-project/outputs/checkpoints/mlm_pretrain_bert/mlm_pretrain_bert_20260416_104251_best
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSIN

,run_name,model,train_domain,eval_name,eval_domain,eval_split,setting,accuracy,macro_f1,weighted_f1,metrics_path,prediction_path,confusion_matrix_path,confusion_figure_path
0,bert_mlm_tfns_20260416_105539,/content/financial-sentiment-project/financial...,tfns,tfns_val,tfns,val,in_domain,0.885149,0.852105,0.885555,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
1,bert_mlm_tfns_20260416_105539,/content/financial-sentiment-project/financial...,tfns,tfns_test,tfns,test,in_domain,0.868644,0.833578,0.869602,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
2,bert_mlm_tfns_20260416_105539,/content/financial-sentiment-project/financial...,tfns,fpb_test,fpb,test,cross_domain,0.817109,0.760277,0.799363,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...


## BERT DAPT (Augmented Corpus): Continued MLM Pretraining + Fine-Tuning

Same setup as above, but the MLM corpus is replaced with
`data/financial_mlm_corpus_20000.csv` (20 000 external financial sentences)
to evaluate whether a broader domain-adaptive corpus improves generalisation.
Expected ~25-35 min on a T4 GPU.

In [12]:
before_ckpts = snapshot_paths("outputs/checkpoints/mlm_pretrain_bert_augmentation/*_best")

run_live_command([
    sys.executable,
    "-m",
    "src.cli.run_mlm_pretrain",
    "--config",
    str(ROOT / "configs/mlm_pretrain_bert_augmentation.yaml"),
])

new_ckpts = newly_created_paths(before_ckpts, "outputs/checkpoints/mlm_pretrain_bert_augmentation/*_best")
mlm_bert_aug_checkpoint_dir = new_ckpts[-1] if new_ckpts else newest_path("outputs/checkpoints/mlm_pretrain_bert_augmentation/*_best")

print(f"BERT MLM (augmented) checkpoint directory: {mlm_bert_aug_checkpoint_dir}")
if mlm_bert_aug_checkpoint_dir is None:
    raise FileNotFoundError("Could not find BERT MLM augmentation checkpoint directory after pretraining.")

$ cd /content/financial-sentiment-project/financial-sentiment-project && PROJECT_ROOT=/content/financial-sentiment-project/financial-sentiment-project PYTHONUNBUFFERED=1 /usr/bin/python3 -m src.cli.run_mlm_pretrain --config /content/financial-sentiment-project/financial-sentiment-project/configs/mlm_pretrain_bert_augmentation.yaml
BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Starting: MLM Pretraining
Training config: epochs=5, batch_size=16, lr=5.00e-05
MLM Pretraining | epoch 1/5 | step 1188/1188 | t

In [13]:
if "mlm_bert_aug_checkpoint_dir" not in globals() or mlm_bert_aug_checkpoint_dir is None:
    mlm_bert_aug_checkpoint_dir = newest_path("outputs/checkpoints/mlm_pretrain_bert_augmentation/*_best")

if mlm_bert_aug_checkpoint_dir is None:
    raise FileNotFoundError("No BERT MLM augmentation checkpoint found. Run the pretraining cell first.")

temp_bert_aug_fpb_config = make_temp_mlm_ft_config(
    ROOT / "configs/transformer_bert_mlm_fpb_augmentation.yaml",
    mlm_bert_aug_checkpoint_dir,
    "auto",
)

temp_bert_aug_tfns_config = make_temp_mlm_ft_config(
    ROOT / "configs/transformer_bert_mlm_tfns_augmentation.yaml",
    mlm_bert_aug_checkpoint_dir,
    "auto",
)

print("BERT MLM augmentation fine-tuning configs ready.")
print(temp_bert_aug_fpb_config)
print(temp_bert_aug_tfns_config)

Temporary config written to: /content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_fpb_augmentation_auto.yaml
Temporary config written to: /content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_tfns_augmentation_auto.yaml
BERT MLM augmentation fine-tuning configs ready.
/content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_fpb_augmentation_auto.yaml
/content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_tfns_augmentation_auto.yaml


In [14]:
before = snapshot_paths("outputs/metrics/mlm_finetune_bert_augmentation/*.json")

run_live_command([
    sys.executable,
    "-m",
    "src.cli.train_transformer",
    "--config",
    str(temp_bert_aug_fpb_config),
])

new_metric_files = newly_created_paths(before, "outputs/metrics/mlm_finetune_bert_augmentation/*.json")
bert_aug_fpb_metrics_path = new_metric_files[-1] if new_metric_files else newest_path("outputs/metrics/mlm_finetune_bert_augmentation/*.json")

print(f"BERT DAPT augmentation+FT (FPB) metrics file: {bert_aug_fpb_metrics_path}")
display(flatten_metrics_json(bert_aug_fpb_metrics_path))

$ cd /content/financial-sentiment-project/financial-sentiment-project && PROJECT_ROOT=/content/financial-sentiment-project/financial-sentiment-project PYTHONUNBUFFERED=1 /usr/bin/python3 -m src.cli.train_transformer --config /content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_fpb_augmentation_auto.yaml
BertForSequenceClassification LOAD REPORT from: /content/financial-sentiment-project/financial-sentiment-project/outputs/checkpoints/mlm_pretrain_bert_augmentation/mlm_pretrain_bert_augmentation_20260416_110434_best
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.d

,run_name,model,train_domain,eval_name,eval_domain,eval_split,setting,accuracy,macro_f1,weighted_f1,metrics_path,prediction_path,confusion_matrix_path,confusion_figure_path
0,bert_mlm_fpb_augmentation_20260416_112349,/content/financial-sentiment-project/financial...,fpb,fpb_val,fpb,val,in_domain,0.976401,0.969266,0.976601,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
1,bert_mlm_fpb_augmentation_20260416_112349,/content/financial-sentiment-project/financial...,fpb,fpb_test,fpb,test,in_domain,0.943953,0.936067,0.944857,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
2,bert_mlm_fpb_augmentation_20260416_112349,/content/financial-sentiment-project/financial...,fpb,tfns_test,tfns,test,cross_domain,0.718220,0.615340,0.706997,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...


In [15]:
before = snapshot_paths("outputs/metrics/mlm_finetune_bert_augmentation/*.json")

run_live_command([
    sys.executable,
    "-m",
    "src.cli.train_transformer",
    "--config",
    str(temp_bert_aug_tfns_config),
])

new_metric_files = newly_created_paths(before, "outputs/metrics/mlm_finetune_bert_augmentation/*.json")
bert_aug_tfns_metrics_path = new_metric_files[-1] if new_metric_files else newest_path("outputs/metrics/mlm_finetune_bert_augmentation/*.json")

print(f"BERT DAPT augmentation+FT (TFNS) metrics file: {bert_aug_tfns_metrics_path}")
display(flatten_metrics_json(bert_aug_tfns_metrics_path))

$ cd /content/financial-sentiment-project/financial-sentiment-project && PROJECT_ROOT=/content/financial-sentiment-project/financial-sentiment-project PYTHONUNBUFFERED=1 /usr/bin/python3 -m src.cli.train_transformer --config /content/financial-sentiment-project/financial-sentiment-project/outputs/tmp_configs/transformer_bert_mlm_tfns_augmentation_auto.yaml
BertForSequenceClassification LOAD REPORT from: /content/financial-sentiment-project/financial-sentiment-project/outputs/checkpoints/mlm_pretrain_bert_augmentation/mlm_pretrain_bert_augmentation_20260416_110434_best
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.

,run_name,model,train_domain,eval_name,eval_domain,eval_split,setting,accuracy,macro_f1,weighted_f1,metrics_path,prediction_path,confusion_matrix_path,confusion_figure_path
0,bert_mlm_tfns_augmentation_20260416_112539,/content/financial-sentiment-project/financial...,tfns,tfns_val,tfns,val,in_domain,0.877577,0.840287,0.878110,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
1,bert_mlm_tfns_augmentation_20260416_112539,/content/financial-sentiment-project/financial...,tfns,tfns_test,tfns,test,in_domain,0.863701,0.826896,0.863975,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
2,bert_mlm_tfns_augmentation_20260416_112539,/content/financial-sentiment-project/financial...,tfns,fpb_test,fpb,test,cross_domain,0.867257,0.855047,0.867936,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...


## Collect and Summarise All DAPT-BERT Results

In [16]:
all_results_df = collect_dapt_metrics()

if all_results_df.empty:
    print("No DAPT-BERT metrics JSON files found under outputs/metrics.")
else:
    display(all_results_df)

summary_out = ROOT / "outputs/metrics/dapt_experiment_manifest.csv"
summary_out.parent.mkdir(parents=True, exist_ok=True)
all_results_df.to_csv(summary_out, index=False)
print(f"DAPT experiment manifest saved to: {summary_out}")

,run_name,model,train_domain,eval_name,eval_domain,eval_split,setting,accuracy,macro_f1,weighted_f1,metrics_path,prediction_path,confusion_matrix_path,confusion_figure_path
0,bert_mlm_fpb_20260416_105113,/content/financial-sentiment-project/financial...,fpb,fpb_test,fpb,test,in_domain,0.979351,0.975231,0.979477,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
1,bert_mlm_fpb_20260416_105113,/content/financial-sentiment-project/financial...,fpb,fpb_val,fpb,val,in_domain,0.976401,0.966415,0.976586,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
2,bert_mlm_fpb_20260416_105113,/content/financial-sentiment-project/financial...,fpb,tfns_test,tfns,test,cross_domain,0.720339,0.672261,0.728330,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
3,bert_mlm_tfns_20260416_105539,/content/financial-sentiment-project/financial...,tfns,fpb_test,fpb,test,cross_domain,0.817109,0.760277,0.799363,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
4,bert_mlm_tfns_20260416_105539,/content/financial-sentiment-project/financial...,tfns,tfns_test,tfns,test,in_domain,0.868644,0.833578,0.869602,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
5,bert_mlm_tfns_20260416_105539,/content/financial-sentiment-project/financial...,tfns,tfns_val,tfns,val,in_domain,0.885149,0.852105,0.885555,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
6,bert_mlm_fpb_augmentation_20260416_112349,/content/financial-sentiment-project/financial...,fpb,fpb_test,fpb,test,in_domain,0.943953,0.936067,0.944857,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
7,bert_mlm_fpb_augmentation_20260416_112349,/content/financial-sentiment-project/financial...,fpb,fpb_val,fpb,val,in_domain,0.976401,0.969266,0.976601,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
8,bert_mlm_fpb_augmentation_20260416_112349,/content/financial-sentiment-project/financial...,fpb,tfns_test,tfns,test,cross_domain,0.718220,0.615340,0.706997,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
9,bert_mlm_tfns_augmentation_20260416_112539,/content/financial-sentiment-project/financial...,tfns,fpb_test,fpb,test,cross_domain,0.867257,0.855047,0.867936,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...


DAPT experiment manifest saved to: /content/financial-sentiment-project/financial-sentiment-project/outputs/metrics/dapt_experiment_manifest.csv


## Filter to Test-Set Results

In [17]:
if "all_results_df" not in globals() or all_results_df.empty:
    all_results_df = collect_dapt_metrics()

test_results_df = all_results_df[all_results_df["eval_split"] == "test"].copy()
test_results_df = test_results_df.sort_values(
    ["model", "train_domain", "eval_domain", "macro_f1"],
    ascending=[True, True, True, False],
).reset_index(drop=True)

display(test_results_df)

,run_name,model,train_domain,eval_name,eval_domain,eval_split,setting,accuracy,macro_f1,weighted_f1,metrics_path,prediction_path,confusion_matrix_path,confusion_figure_path
0,bert_mlm_fpb_20260416_105113,/content/financial-sentiment-project/financial...,fpb,fpb_test,fpb,test,in_domain,0.979351,0.975231,0.979477,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
1,bert_mlm_fpb_20260416_105113,/content/financial-sentiment-project/financial...,fpb,tfns_test,tfns,test,cross_domain,0.720339,0.672261,0.728330,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
2,bert_mlm_tfns_20260416_105539,/content/financial-sentiment-project/financial...,tfns,fpb_test,fpb,test,cross_domain,0.817109,0.760277,0.799363,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
3,bert_mlm_tfns_20260416_105539,/content/financial-sentiment-project/financial...,tfns,tfns_test,tfns,test,in_domain,0.868644,0.833578,0.869602,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
4,bert_mlm_fpb_augmentation_20260416_112349,/content/financial-sentiment-project/financial...,fpb,fpb_test,fpb,test,in_domain,0.943953,0.936067,0.944857,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
5,bert_mlm_fpb_augmentation_20260416_112349,/content/financial-sentiment-project/financial...,fpb,tfns_test,tfns,test,cross_domain,0.718220,0.615340,0.706997,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
6,bert_mlm_tfns_augmentation_20260416_112539,/content/financial-sentiment-project/financial...,tfns,fpb_test,fpb,test,cross_domain,0.867257,0.855047,0.867936,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...
7,bert_mlm_tfns_augmentation_20260416_112539,/content/financial-sentiment-project/financial...,tfns,tfns_test,tfns,test,in_domain,0.863701,0.826896,0.863975,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...,/content/financial-sentiment-project/financial...


## Compute Generalisation Drop

In [18]:
if "test_results_df" not in globals():
    test_results_df = collect_dapt_metrics()
    test_results_df = test_results_df[test_results_df["eval_split"] == "test"].copy()

rows = []
for run_name, group in test_results_df.groupby("run_name"):
    train_domain = group["train_domain"].iloc[0]
    if train_domain == "none":
        continue

    in_domain = group[group["eval_domain"] == train_domain]
    cross_domain = group[group["eval_domain"] != train_domain]

    if in_domain.empty or cross_domain.empty:
        continue

    in_macro_f1 = float(in_domain.iloc[0]["macro_f1"])

    for _, row in cross_domain.iterrows():
        rows.append(
            {
                "run_name": run_name,
                "model": row["model"],
                "train_domain": train_domain,
                "cross_eval_domain": row["eval_domain"],
                "in_domain_macro_f1": in_macro_f1,
                "cross_domain_macro_f1": float(row["macro_f1"]),
                "generalisation_drop": in_macro_f1 - float(row["macro_f1"]),
            }
        )

generalisation_drop_df = pd.DataFrame(rows)
display(generalisation_drop_df)

,run_name,model,train_domain,cross_eval_domain,in_domain_macro_f1,cross_domain_macro_f1,generalisation_drop
0,bert_mlm_fpb_20260416_105113,/content/financial-sentiment-project/financial...,fpb,tfns,0.975231,0.672261,0.302969
1,bert_mlm_fpb_augmentation_20260416_112349,/content/financial-sentiment-project/financial...,fpb,tfns,0.936067,0.615340,0.320726
2,bert_mlm_tfns_20260416_105539,/content/financial-sentiment-project/financial...,tfns,fpb,0.833578,0.760277,0.073302
3,bert_mlm_tfns_augmentation_20260416_112539,/content/financial-sentiment-project/financial...,tfns,fpb,0.826896,0.855047,-0.028152


In [19]:
import shutil

# Lightweight outputs only: metrics / figures / logs — no model checkpoints
lightweight_sources = [
    (ROOT / "outputs/metrics/mlm_finetune_bert",                "metrics/mlm_finetune_bert"),
    (ROOT / "outputs/metrics/mlm_finetune_bert_augmentation",   "metrics/mlm_finetune_bert_augmentation"),
    (ROOT / "outputs/figures/mlm_finetune_bert",                "figures/mlm_finetune_bert"),
    (ROOT / "outputs/figures/mlm_finetune_bert_augmentation",   "figures/mlm_finetune_bert_augmentation"),
    (ROOT / "outputs/logs/mlm_finetune_bert",                   "logs/mlm_finetune_bert"),
    (ROOT / "outputs/logs/mlm_pretrain_bert",                   "logs/mlm_pretrain_bert"),
    (ROOT / "outputs/logs/mlm_finetune_bert_augmentation",      "logs/mlm_finetune_bert_augmentation"),
    (ROOT / "outputs/logs/mlm_pretrain_bert_augmentation",      "logs/mlm_pretrain_bert_augmentation"),
]

if IN_COLAB:
    drive_out = Path("/content/drive/MyDrive/bert-dapt-outputs")
else:
    drive_out = ROOT / "outputs/saved_results/bert-dapt-outputs"

drive_out.mkdir(parents=True, exist_ok=True)

for src, rel_dst in lightweight_sources:
    if src.exists():
        dst = drive_out / rel_dst
        if dst.exists():
            shutil.rmtree(dst)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst)
        print(f"Saved: {rel_dst}")
    else:
        print(f"Skip (not found): {rel_dst}")

print(f"\nAll lightweight outputs saved to: {drive_out}")

# Package as zip and trigger browser download
zip_path = Path("/content/bert_dapt_outputs") if IN_COLAB else ROOT / "outputs/bert_dapt_outputs"
if zip_path.with_suffix(".zip").exists():
    zip_path.with_suffix(".zip").unlink()
shutil.make_archive(str(zip_path), "zip", str(drive_out.parent), drive_out.name)
print(f"Zip created: {zip_path}.zip")

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path) + ".zip")
    print("Browser download triggered.")
else:
    print(f"Zip ready at: {zip_path}.zip")

Saved: metrics/mlm_finetune_bert
Saved: metrics/mlm_finetune_bert_augmentation
Saved: figures/mlm_finetune_bert
Saved: figures/mlm_finetune_bert_augmentation
Saved: logs/mlm_finetune_bert
Saved: logs/mlm_pretrain_bert
Saved: logs/mlm_finetune_bert_augmentation
Saved: logs/mlm_pretrain_bert_augmentation

All lightweight outputs saved to: /content/drive/MyDrive/bert-dapt-outputs
Zip created: /content/bert_dapt_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Browser download triggered.
